# Trading APP


This notebook now runs the full process end to end:

1. Train the traditional ML live-trade stack.
2. Save the classifier, regressor, and autoencoder artifacts.
3. Build and save the latest leaderboard.
4. Build and save historical entry-trade embeddings for similar-trade search.
5. Run the similar-trades lookup for a chosen symbol using those saved artifacts.
6. Launch the Streamlit trading app for the app UI.


In [1]:
import importlib
import os
import sys
import subprocess
import threading
import time
import urllib.request
from datetime import datetime
from pathlib import Path
from types import SimpleNamespace

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

def _find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "app").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError(f"Could not locate repo root from cwd={start}")

def _load_repo_env(repo_root: Path) -> None:
    env_path = repo_root / ".env"
    if not env_path.exists():
        return
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = str(key).strip()
        if not key:
            continue
        os.environ[key] = str(value).strip().strip('"').strip("'")

def _bootstrap_repo(repo_root: Path) -> None:
    os.chdir(repo_root)
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))
    _load_repo_env(repo_root)
    os.environ.setdefault("DJANGO_ALLOW_ASYNC_UNSAFE", "true")
    os.environ.setdefault("DJANGO_SETTINGS_MODULE", "settings")

def _reload_app_modules():
    import app.live_trade_leaderboard as live_trade_leaderboard
    import app.optimal_trade_lookup as optimal_trade_lookup
    return (
        importlib.reload(live_trade_leaderboard),
        importlib.reload(optimal_trade_lookup),
    )

_NB_LAST_LOG_TS = time.time()

def _nb_log(message: str) -> None:
    global _NB_LAST_LOG_TS
    _NB_LAST_LOG_TS = time.time()
    stamp = datetime.now().strftime("%H:%M:%S")
    print(f"[{stamp}] {message}", flush=True)

def run_with_progress(label, fn, *args, heartbeat_seconds=30, **kwargs):
    stop_event = threading.Event()
    start = time.time()

    def _heartbeat():
        while not stop_event.wait(max(int(heartbeat_seconds), 1)):
            idle_seconds = time.time() - _NB_LAST_LOG_TS
            if idle_seconds < max(int(heartbeat_seconds), 1):
                continue
            elapsed = time.time() - start
            _nb_log(f"{label} still running... elapsed {elapsed / 60.0:.1f} min")

    _nb_log(f"Starting: {label}")
    thread = None
    if heartbeat_seconds and int(heartbeat_seconds) > 0:
        thread = threading.Thread(target=_heartbeat, daemon=True)
        thread.start()
    try:
        value = fn(*args, **kwargs)
    except Exception:
        elapsed = time.time() - start
        _nb_log(f"Failed: {label} after {elapsed / 60.0:.1f} min")
        stop_event.set()
        if thread is not None:
            thread.join(timeout=1.0)
        raise
    elapsed = time.time() - start
    _nb_log(f"Finished: {label} in {elapsed / 60.0:.1f} min")
    stop_event.set()
    if thread is not None:
        thread.join(timeout=1.0)
    return value

_STREAMLIT_PROCESS = None

def _url_responds(url: str, *, timeout: float = 1.0) -> bool:
    try:
        with urllib.request.urlopen(url, timeout=timeout) as response:
            return int(response.status) < 500
    except Exception:
        return False

def start_streamlit_trading_app(*, host: str = "localhost", port: int = 8502) -> dict[str, str]:
    global _STREAMLIT_PROCESS
    root_url = f"http://{host}:{int(port)}"
    app_path = repo_root / "app" / "streamlit_optimal_trade_finder.py"
    if _url_responds(root_url):
        _nb_log(f"Streamlit trading app already responding: {root_url}")
    else:
        _nb_log(f"Starting Streamlit trading app: {app_path} on port {port}")
        _STREAMLIT_PROCESS = subprocess.Popen(
            [
                sys.executable,
                "-m",
                "streamlit",
                "run",
                str(app_path),
                "--server.port",
                str(int(port)),
                "--server.headless",
                "true",
            ],
            cwd=str(repo_root),
            env=os.environ.copy(),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
        )
        for _ in range(30):
            if _url_responds(root_url):
                break
            time.sleep(1)
        else:
            raise RuntimeError(f"Streamlit did not respond at {root_url}. Check _STREAMLIT_PROCESS output.")
    urls = {
        "streamlit_app": root_url,
        "leaderboard_page": root_url,
    }
    _nb_log(f"Streamlit trading app: {urls['streamlit_app']}")
    return urls

repo_root = _find_repo_root(Path.cwd())
_bootstrap_repo(repo_root)
live_trade_leaderboard, optimal_trade_lookup = _reload_app_modules()

from app.live_trade_leaderboard import default_live_trade_config, load_saved_leaderboard, run_live_trade_leaderboard_build
from app.optimal_trade_lookup import OptimalTradeQuery, find_nearest_optimal_trades


def _expected_latest_price_date_from_market_clock():
    now_et = pd.Timestamp.now(tz="America/New_York")
    if now_et.weekday() < 5 and now_et.hour >= 17:
        return now_et.date()
    return (now_et.normalize() - pd.offsets.BDay(1)).date()

_nb_log(f"Notebook bootstrap complete. Repo root: {repo_root}")
_nb_log(f"Working directory set to: {Path.cwd().resolve()}")
_nb_log(f"Python executable: {sys.executable}")
_nb_log(f"DJANGO_SETTINGS_MODULE={os.environ.get('DJANGO_SETTINGS_MODULE')}")


def _load_recent_similarity_build(cfg: dict, *, max_age: pd.Timedelta = pd.Timedelta(days=1)):
    artifact_dir = Path(str(cfg["runtime"]["artifact_dir"])).expanduser().resolve()
    required_paths = [
        artifact_dir / "meta.json",
        artifact_dir / "clf_raw.pkl",
        artifact_dir / "reg_trade_return_raw.pkl",
        artifact_dir / "ae_raw.pkl",
        artifact_dir / "leaderboard_latest_meta.json",
        artifact_dir / "leaderboard_latest.pkl",
    ]
    missing = [path.name for path in required_paths if not path.exists()]
    if missing:
        return None, {
            "artifact_dir": str(artifact_dir),
            "age": pd.NaT,
            "reason": f"missing_files={','.join(missing)}",
        }

    latest_mtime = max(path.stat().st_mtime for path in required_paths)
    artifact_timestamp = pd.Timestamp(latest_mtime, unit="s", tz="UTC")
    artifact_age = pd.Timestamp.now(tz="UTC") - artifact_timestamp
    if artifact_age > pd.Timedelta(max_age):
        return None, {
            "artifact_dir": str(artifact_dir),
            "age": artifact_age,
            "reason": f"older_than_{pd.Timedelta(max_age)}",
        }

    saved = load_saved_leaderboard(artifact_dir=str(artifact_dir))
    if saved is None:
        return None, {
            "artifact_dir": str(artifact_dir),
            "age": artifact_age,
            "reason": "missing_saved_leaderboard",
        }

    leaderboard, meta = saved
    meta = dict(meta or {})
    latest_date = pd.Timestamp(meta.get("latest_date") or pd.Timestamp.today().normalize()).normalize()
    requested_end_date = pd.Timestamp(str(cfg["dates"].get("data_end") or pd.Timestamp.today().date())).normalize()
    expected_score_date = min(
        requested_end_date,
        pd.Timestamp(_expected_latest_price_date_from_market_clock()).normalize(),
    )
    if latest_date < expected_score_date:
        return None, {
            "artifact_dir": str(artifact_dir),
            "age": artifact_age,
            "reason": (
                f"saved_latest_date_{latest_date.date().isoformat()}_lt_"
                f"expected_{expected_score_date.date().isoformat()}"
            ),
            "saved_latest_date": latest_date.date().isoformat(),
            "expected_score_date": expected_score_date.date().isoformat(),
        }
    universe_size = int(pd.to_numeric(pd.Series([meta.get("universe_size")]), errors="coerce").fillna(0).iloc[0])
    vector_metadata = dict(meta.get("vector_metadata") or {})
    reference_trade_count = int(pd.to_numeric(pd.Series([vector_metadata.get("row_count")]), errors="coerce").fillna(0).iloc[0])
    build_result = SimpleNamespace(
        latest_date=latest_date,
        artifact_dir=artifact_dir,
        leaderboard=leaderboard,
        universe=tuple(range(universe_size)),
        reference_trade_count=reference_trade_count,
        vector_metadata=vector_metadata,
        source="artifacts",
    )
    return build_result, {
        "artifact_dir": str(artifact_dir),
        "age": artifact_age,
        "reason": "fresh_saved_build",
    }



[10:06:58] Notebook bootstrap complete. Repo root: /Users/johnnylee/PycharmProjects/optimal_trader
[10:06:58] Working directory set to: /Users/johnnylee/PycharmProjects/optimal_trader
[10:06:58] Python executable: /Users/johnnylee/miniconda3/envs/optimal_trader/bin/python3.13
[10:06:58] DJANGO_SETTINGS_MODULE=settings


In [2]:
QUERY_SYMBOL = "AAPL"
SIMILAR_TRADES_TOP_K = 10
LEADERBOARD_TOP_K = 20
DATA_START = "1900-01-01"
MIN_MARKET_CAP = 10_000_000_000.0
REFRESH_FMP_DATA = True
REFRESH_MACRO_DATA = True

def make_notebook_config() -> dict:
    cfg = default_live_trade_config()
    cfg["dates"]["data_start"] = DATA_START
    cfg["dates"]["data_end"] = pd.Timestamp.today().strftime("%Y-%m-%d")
    cfg["universe"]["min_market_cap"] = MIN_MARKET_CAP
    artifact_dir = (repo_root / "artifacts" / "raw_stack").resolve()
    artifact_dir.mkdir(parents=True, exist_ok=True)
    cfg["runtime"]["artifact_dir"] = str(artifact_dir)
    cfg["fmp_refresh"]["enabled"] = REFRESH_FMP_DATA
    cfg["fmp_refresh"]["refresh_symbol_sections_before_build"] = REFRESH_FMP_DATA
    cfg["fmp_refresh"]["refresh_macro_before_build"] = REFRESH_MACRO_DATA
    cfg["fmp_refresh"]["verbose"] = False
    cfg["strategy"]["top_k"] = LEADERBOARD_TOP_K
    return cfg

def notebook_run_summary(cfg: dict) -> dict:
    return {
        "query_symbol": QUERY_SYMBOL,
        "data_start": cfg["dates"]["data_start"],
        "data_end": cfg["dates"]["data_end"],
        "min_market_cap": cfg["universe"]["min_market_cap"],
        "download_missing_fmp_data": cfg["fmp_refresh"]["refresh_symbol_sections_before_build"],
        "refresh_macro_data": cfg["fmp_refresh"]["refresh_macro_before_build"],
        "leaderboard_top_k": cfg["strategy"]["top_k"],
        "similar_trades_top_k": SIMILAR_TRADES_TOP_K,
        "artifact_dir": cfg["runtime"]["artifact_dir"],
    }

APP_CFG = make_notebook_config()
_nb_log("Configuration prepared for end-to-end run.")
_nb_log(f"Artifact dir: {APP_CFG['runtime']['artifact_dir']}")
_nb_log(f"Universe min market cap: ${APP_CFG['universe']['min_market_cap']:,.0f}")
_nb_log(f"Download missing FMP symbol data: {APP_CFG['fmp_refresh']['refresh_symbol_sections_before_build']}")
_nb_log(f"Leaderboard top_k: {APP_CFG['strategy']['top_k']}")
_nb_log(f"Similar-trades query symbol: {QUERY_SYMBOL}")

display(pd.DataFrame([notebook_run_summary(APP_CFG)]))


[10:06:58] Configuration prepared for end-to-end run.
[10:06:58] Artifact dir: /Users/johnnylee/PycharmProjects/optimal_trader/artifacts/raw_stack
[10:06:58] Universe min market cap: $10,000,000,000
[10:06:58] Download missing FMP symbol data: True
[10:06:58] Leaderboard top_k: 20
[10:06:58] Similar-trades query symbol: AAPL


,query_symbol,data_start,data_end,min_market_cap,download_missing_fmp_data,refresh_macro_data,leaderboard_top_k,similar_trades_top_k,artifact_dir
0,AAPL,1900-01-01,2026-05-26,1.000000e+10,True,True,20,10,/Users/johnnylee/PycharmProjects/optimal_trade...


In [3]:
recent_build_result, artifact_status = _load_recent_similarity_build(APP_CFG, max_age=pd.Timedelta(days=1))
model_source = "retrained"
if recent_build_result is not None:
    build_result = recent_build_result
    model_source = "artifacts"
    _nb_log(
        "Reusing saved raw-stack + leaderboard artifacts from "
        f"{artifact_status['artifact_dir']} | age {artifact_status['age'] / pd.Timedelta(hours=1):.2f}h"
    )
else:
    _nb_log(
        "Saved artifacts were not reusable; running full leaderboard build | "
        f"reason={artifact_status.get('reason') or 'unknown'}"
    )
    build_result = run_with_progress(
        "Train models, save artifacts, build leaderboard, and write trade vectors",
        run_live_trade_leaderboard_build,
        config=APP_CFG,
        progress_logger=_nb_log,
        heartbeat_seconds=0,
    )

_nb_log(f"Latest scored date: {pd.Timestamp(build_result.latest_date).date().isoformat()}")
_nb_log(f"Universe size: {len(build_result.universe):,}")
_nb_log(f"Historical entry trades embedded: {build_result.reference_trade_count:,}")
_nb_log(f"Vector backend: {str(build_result.vector_metadata.get('backend') or '')}")

leaderboard_summary = pd.DataFrame([
    {
        "latest_date": pd.Timestamp(build_result.latest_date).date().isoformat(),
        "universe_size": len(build_result.universe),
        "leaderboard_rows": len(build_result.leaderboard),
        "reference_trade_count": build_result.reference_trade_count,
        "artifact_dir": str(build_result.artifact_dir),
        "vector_backend": str(build_result.vector_metadata.get("backend") or ""),
        "model_source": model_source,
        "artifact_age_hours": None if pd.isna(artifact_status.get("age")) else round(float(artifact_status["age"] / pd.Timedelta(hours=1)), 2),
        "artifact_status": str(artifact_status.get("reason") or ""),
        "saved_latest_date": str(artifact_status.get("saved_latest_date") or ""),
        "expected_score_date": str(artifact_status.get("expected_score_date") or ""),
    }
])

display(leaderboard_summary)
display(build_result.leaderboard.head(25))


[10:06:58] Reusing saved raw-stack + leaderboard artifacts from /Users/johnnylee/PycharmProjects/optimal_trader/artifacts/raw_stack | age 0.45h
[10:06:58] Latest scored date: 2026-05-25
[10:06:58] Universe size: 694
[10:06:58] Historical entry trades embedded: 202,400
[10:06:58] Vector backend: faiss


,latest_date,universe_size,leaderboard_rows,reference_trade_count,artifact_dir,vector_backend,model_source,artifact_age_hours,artifact_status,saved_latest_date,expected_score_date
0,2026-05-25,694,694,202400,/Users/johnnylee/PycharmProjects/optimal_trade...,faiss,artifacts,0.45,fresh_saved_build,,


,Rank,Scored Date,Symbol,Direction,Eligible,Classifier Score,Regressor Score,Autoencoder Score,Combined Score,Similar Trades,__price,__long_score,__short_score,__score_col_name,__component__0,__component__1,__component__2,__component__3,__component__4,__component__5
0,1,2026-05-22,CRWD,Short,True,0.988017,0.786128,1.000000,0.943866,/?symbol=CRWD,663.46,0.637341,0.943866,short_score_mean_raw_pct6,0.988017,0.786128,1.000000,0.932277,0.974063,0.982709
1,2,2026-05-22,CSCO,Short,True,0.979758,0.713605,1.000000,0.913951,/?symbol=CSCO,120.41,0.628912,0.913951,short_score_mean_raw_pct6,0.979758,0.713605,1.000000,0.876081,0.958213,0.956052
2,3,2026-05-22,QCOM,Short,True,0.957324,0.766594,1.000000,0.907637,/?symbol=QCOM,238.16,0.665138,0.907637,short_score_mean_raw_pct6,0.957324,0.766594,1.000000,0.770893,0.968300,0.982709
3,4,2026-05-22,HPE,Short,True,0.926792,0.848641,1.000000,0.905416,/?symbol=HPE,37.58,0.701912,0.905416,short_score_mean_raw_pct6,0.926792,0.848641,1.000000,0.684438,0.989914,0.982709
4,5,2026-05-22,MSTR,Long,True,0.977412,0.649858,0.997291,0.904286,/?symbol=MSTR,159.89,0.904286,0.608501,buy_score_mean_raw_pct6,0.977412,0.649858,0.997291,0.910663,0.938040,0.952450
5,6,2026-05-22,RPRX,Short,True,0.985362,0.607853,1.000000,0.897020,/?symbol=RPRX,54.50,0.597144,0.897020,short_score_mean_raw_pct6,0.985362,0.607853,1.000000,0.914986,0.910663,0.963256
6,7,2026-05-22,HPQ,Short,True,0.918150,0.822384,1.000000,0.890834,/?symbol=HPQ,25.24,0.699817,0.890834,short_score_mean_raw_pct6,0.918150,0.822384,1.000000,0.655620,0.988473,0.960375
7,8,2026-05-22,PAA,Short,True,0.979989,0.598287,1.000000,0.890087,/?symbol=PAA,24.15,0.604491,0.890087,short_score_mean_raw_pct6,0.979989,0.598287,1.000000,0.877522,0.902017,0.982709
8,9,2026-05-22,WMS,Long,True,0.992465,0.503245,1.000000,0.872964,/?symbol=WMS,133.00,0.872964,0.550068,buy_score_mean_raw_pct6,0.992465,0.503245,1.000000,0.976945,0.782421,0.982709
9,10,2026-05-22,F,Short,True,0.908210,0.771403,0.996880,0.872595,/?symbol=F,14.93,0.690176,0.872595,short_score_mean_raw_pct6,0.908210,0.771403,0.996880,0.639769,0.971182,0.948127


In [4]:
import os
print(bool(os.getenv("ROBINHOOD_USERNAME")), bool(os.getenv("ROBINHOOD_PASSWORD")))

True True


In [5]:
def make_similarity_query(build_result) -> OptimalTradeQuery:
    latest_date = pd.Timestamp(build_result.latest_date).normalize()
    return OptimalTradeQuery(
        symbol=QUERY_SYMBOL,
        as_of_date=latest_date.strftime("%Y-%m-%d"),
        query_lookback_years=0,
        reference_symbols=(),
        reference_start_date=APP_CFG["dates"]["data_start"],
        reference_end_date=(latest_date - pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
        top_k=SIMILAR_TRADES_TOP_K,
        label_freq="YE",
        label_k_values=tuple(int(v) for v in APP_CFG["labels"]["k_params"]["YE"]),
        min_profit_pct_points=0.0,
        download_missing_prices=False,
        artifact_dir=str(build_result.artifact_dir),
    )

query = make_similarity_query(build_result)
query


OptimalTradeQuery(symbol='AAPL', as_of_date='2026-05-25', query_lookback_years=0, reference_symbols=(), reference_start_date='1900-01-01', reference_end_date='2026-05-24', top_k=10, label_freq='YE', label_k_values=(1, 2, 4, 8), min_profit_pct_points=0.0, download_missing_prices=False, artifact_dir='/Users/johnnylee/PycharmProjects/optimal_trader/artifacts/raw_stack', db_artifact_name='', db_artifact_version=None)

In [6]:
_nb_log(f"Running similar-trades search for {query.symbol} as of {query.as_of_date} with top_k={query.top_k}")
result = run_with_progress(
    f"Find nearest optimal trades for {query.symbol}",
    find_nearest_optimal_trades,
    query,
    heartbeat_seconds=15,
)

_nb_log(f"Nearest trades found: {len(result.nearest_trades):,}")
_nb_log(f"Search method: {result.metadata.get('search_method')}")
_nb_log(f"Lookup source: {result.metadata.get('embedding_source', result.metadata.get('trade_vector_backend', ''))}")

display(result.query_summary)
display(result.nearest_trades)
result.metadata

[10:06:58] Running similar-trades search for AAPL as of 2026-05-25 with top_k=10
[10:06:58] Starting: Find nearest optimal trades for AAPL
[17:07:03] Finished: Find nearest optimal trades for AAPL in 0.1 min
[17:07:03] Nearest trades found: 10
[17:07:03] Search method: rebuilt_reference_trade_catalog
[17:07:03] Lookup source: 


,symbol,as_of_date,ae_familiarity,artifact_source,artifact_numeric_feature_count,reference_trade_count
0,AAPL,2026-05-22,0.721024,/Users/johnnylee/PycharmProjects/optimal_trade...,195,422


,Symbol,Side,Entry Date,Exit Date,Signed Trade Return,Hold Days,AE Familiarity,Entry Price,Exit Price
0,AAPL,short,2024-07-16,2024-08-05,-10.68,20,0.721342,233.02,207.66
1,AAPL,short,2020-09-02,2020-09-21,-16.03,19,0.731015,127.56,106.86
2,AAPL,short,2025-12-03,2025-12-23,-3.95,20,0.751388,283.89,272.11
3,AAPL,short,2022-01-04,2022-06-16,-27.23,163,0.769627,175.82,127.60
4,AAPL,short,2022-08-18,2022-12-30,-25.07,134,0.786568,171.10,127.87
5,AAPL,short,2021-01-27,2021-03-08,-17.77,40,0.774274,138.14,113.32
6,AAPL,short,2021-01-26,2021-03-08,-18.40,41,0.766511,139.21,113.32
7,AAPL,short,2024-08-30,2024-09-16,-5.34,17,0.777750,227.51,214.91
8,AAPL,long,2021-12-20,2021-12-28,5.42,8,0.776894,166.09,175.42
9,AAPL,short,2023-08-01,2023-08-18,-10.47,17,0.780239,193.07,172.46


{'artifact_source': '/Users/johnnylee/PycharmProjects/optimal_trader/artifacts/raw_stack/ae_raw.pkl',
 'artifact_metadata': {'artifact_version': 1,
  'stack': 'raw',
  'feature_list': ['open',
   'high',
   'low',
   'close',
   'volume',
   'px__ret_1d',
   'px__ret_2d',
   'px__ret_3d',
   'px__ret_5d',
   'px__ret_10d',
   'px__ret_20d',
   'px__ret_21d',
   'px__ret_63d',
   'px__ret_126d',
   'px__ret_189d',
   'px__ret_252d',
   'px__dist_sma_5',
   'px__sma_slope_5',
   'px__dist_sma_10',
   'px__sma_slope_10',
   'px__dist_sma_20',
   'px__sma_slope_20',
   'px__dist_sma_50',
   'px__sma_slope_50',
   'px__dist_sma_100',
   'px__sma_slope_100',
   'px__dist_sma_200',
   'px__sma_slope_200',
   'px__dist_ema_12',
   'px__dist_ema_26',
   'px__dist_ema_50',
   'px__macd',
   'px__macd_signal',
   'px__macd_hist',
   'px__z_close_10',
   'px__bb_pos_10',
   'px__z_close_20',
   'px__bb_pos_20',
   'px__z_close_63',
   'px__bb_pos_63',
   'px__hl_range',
   'px__oc_change',
   'px_

## Open Streamlit Trading App

Start the Streamlit trading app and open the URL below. The Streamlit app includes the leaderboard, target sheet, and Robinhood option workflow.


In [7]:
streamlit_urls = start_streamlit_trading_app(port=8502)
streamlit_urls


[17:07:03] Streamlit trading app already responding: http://localhost:8502
[17:07:03] Streamlit trading app: http://localhost:8502


{'streamlit_app': 'http://localhost:8502',
 'leaderboard_page': 'http://localhost:8502'}

## Notes

- The leaderboard build reuses the traditional ML live-trade training path.
- It saves the raw-stack model artifacts into `artifacts/raw_stack`.
- It also saves historical entry-trade latent vectors so the similar-trades page can search faster.
- The similar-trades query uses those saved artifacts instead of assuming they already existed.